In [ ]:
from pixell import enmap, enplot, wcsutils
from pspy import so_map_preprocessing
from mnms import utils

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# noise maps
# ivar for mask

so_types = ['type1', 'type2', 'type3']
so_mapnames = ['all_f090', 'all_f150', 'uhf_f220', 'uhf_f280']
so_map_dir = '/home/zatkins/scratch/projects/lat-iso/phase1/so_maps/sigurdkn/scantest/test3'

so_maps = [enmap.read_map(f'{so_map_dir}/{so_type}_{so_mapname}_sky_map.fits') for so_type in so_types for so_mapname in so_mapnames for i in range(1)]
so_maps = enmap.samewcs(so_maps, so_maps[0])
so_maps = so_maps.reshape(len(so_types), len(so_mapnames), -1, *so_maps.shape[-3:])

so_ivars = [enmap.read_map(f'{so_map_dir}/{so_type}_{so_mapname}_sky_ivar.fits') for so_type in so_types for so_mapname in so_mapnames for i in range(1)]
so_ivars = enmap.samewcs(so_ivars, so_ivars[0])
so_ivars = so_ivars.reshape(len(so_types), len(so_mapnames), -1, 1, *so_ivars.shape[-2:])

In [ ]:
temp = so_ivars[0, 0].squeeze()
temp = enmap.smooth_gauss(temp, np.deg2rad(1))
bottom_half = enmap.ndmap(np.zeros_like(temp), temp.wcs)
bottom_half[:int(bottom_half.shape[0] * 2/3)]=1
mask = np.logical_and((temp > 5.2e-5), bottom_half)

# enmap.write_map(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/phase1/so_maps/sigurdkn/scantest/bool_mask_all.fits', mask.astype(np.int8))

In [ ]:
enplot.pshow(mask, downgrade=16, colorbar=True, ticks=5)

In [ ]:
mask_apo = enmap.apod_mask(mask, np.deg2rad(2))
for t in range(3):
    for f in range(4):
        print(f'{t=}, {f=}')
        enplot.pshow(so_ivars[t, f, 0, 0] * mask + so_ivars[t, f, 0, 0], downgrade=16, colorbar=True, ticks=5)

for t in range(3):
    for f in range(4):
        for p in range(3):
            print(f'{t=}, {f=}, {p=}')
            enplot.pshow(so_maps[t, f, 0, p] * mask_apo, downgrade=16, colorbar=True, ticks=5, range=[600, 300, 300][p])

In [ ]:
# for noise and signal

act_map_dir = '/home/zatkins/scratch/projects/lat-iso/phase1/act_maps'

act_planck_map = enmap.read_map(f'{act_map_dir}/act-planck_dr6.02_coadd_AA_night_f090_map.fits')
act_planck_map = act_planck_map.reshape(1, 1, *act_planck_map.shape)
act_planck_map = enmap.extract(act_planck_map, *mask.geometry)

# act_maps = [enmap.read_map(f'{act_map_dir}/act_dr6.02_std_AA_night_pa5_f090_4way_set{i}_map_srcfree.fits') for i in range(4)]
# act_maps = enmap.samewcs(act_maps, act_maps[0])
# act_maps = act_maps.reshape(1, *act_maps.shape)
# act_maps = enmap.extract(act_maps, *mask.geometry)

w_act = enmap.read_map(f'{act_map_dir}/window_dr6_pa5_f090_baseline.fits')
w_act = enmap.extract(w_act, *mask.geometry)

In [ ]:
def get_2d_noise_ps(map_diff, box, mask_obs, mask_hole, apod_deg=1):
    box = np.deg2rad(box)

    map_diff = map_diff.submap(box)
    square_mask = enmap.submap(enmap.ones(*mask_obs.geometry, mask_obs.dtype), box)
    square_mask *= mask_obs.submap(box)
    square_mask = enmap.apod_mask(square_mask, np.deg2rad(apod_deg))
    square_mask *= mask_hole.submap(box)

    lwcs = enmap.lwcs(map_diff.shape, map_diff.wcs)

    ps_list = []

    kmap = enmap.fft(map_diff * square_mask)

    for kpol in range(3):
        ps = np.real(kmap[kpol] * kmap[kpol].conj())

        if kpol in (0, 1):
            ps_list.append(ps)
        else:
            ps_list[1] += ps
            ps_list[1] /=2

    ps = enmap.ndmap(ps_list, lwcs)

    return ps, square_mask

def get_flat_2d_ps(ps, modlmap, bin_edges):
    out = np.zeros_like(ps)
    for idx in np.ndindex(ps.shape[:-2]):
        out[idx] = ps[idx] / utils.interp1d_bins(bin_edges, utils.radial_bin(ps[idx], modlmap, bin_edges), bounds_error=False)(modlmap)
    return out

In [ ]:
plot_so_types = ['type1', 'type3']
plot_so_mapname = 'all_f090'

apod_deg = 1
box = [[-25, -12.5], [-15, -22.5]]

noise = {}
flat_noise = {}
for so_type in plot_so_types:

    t = so_types.index(so_type)
    f = so_mapnames.index(plot_so_mapname)

    noise[so_type], square_mask = get_2d_noise_ps(so_maps[t, f].squeeze() - act_planck_map.squeeze(), box, mask, w_act, apod_deg=apod_deg)

    modlmap = square_mask.modlmap().astype(np.float32)
    bin_edges = np.arange(0, 20_000, 200)
    flat_noise[so_type] = get_flat_2d_ps(noise[so_type], modlmap, bin_edges)

In [ ]:
enplot.pshow(1*(so_maps[t, f].squeeze()[0].submap(np.deg2rad(box)) - act_planck_map.squeeze()[0].submap(np.deg2rad(box))), ticks=1, colorbar=True, downgrade=2)
enplot.pshow(square_mask*(so_maps[t, f].squeeze()[0].submap(np.deg2rad(box)) - act_planck_map.squeeze()[0].submap(np.deg2rad(box))), ticks=1, colorbar=True, downgrade=2)

In [ ]:
enplot.pshow(square_mask, ticks=1, colorbar=True, downgrade=2)

pol = 1

# center
for so_type in plot_so_types:
    print(so_type)
    enplot.pshow(utils.crop_center(enmap.fftshift(flat_noise[so_type][pol]), 30), ticks=[50, 90], colorbar=True, upgrade=(20, 20), min=-1, max=3)

# zoom out
downgrade = 4
for so_type in plot_so_types:
    print(so_type)
    enplot.pshow(utils.crop_center(enmap.fftshift(flat_noise[so_type][pol]), 600).downgrade(downgrade), ticks=[3000], colorbar=True, upgrade=(downgrade, downgrade), min=0.25, max=1.75)

In [ ]:
def get_2d_signal_ps(map_auto, box, mask_obs, mask_hole, apod_deg=1):
    box = np.deg2rad(box)

    map_auto = map_auto.submap(box)
    square_mask = enmap.submap(enmap.ones(*mask_obs.geometry, mask_obs.dtype), box)
    square_mask *= mask_obs.submap(box)
    square_mask = enmap.apod_mask(square_mask, np.deg2rad(apod_deg))
    square_mask *= mask_hole.submap(box)

    lwcs = enmap.lwcs(map_auto.shape, map_auto.wcs)

    ps_list = []

    kmap = enmap.fft(map_auto * square_mask)

    for kpol in range(3):
        ps = 0
        ct = 0
        for s1 in range(kmap.shape[-4]):
            for s2 in range(s1+1, kmap.shape[-4]):
                ps += np.real(kmap[s1, kpol] * kmap[s2, kpol].conj())
                ct += 1
        ps /= ct

        if kpol in (0, 1):
            ps_list.append(ps)
        else:
            ps_list[1] += ps
            ps_list[1] /=2

    ps = enmap.ndmap(ps_list, lwcs)

    return ps, square_mask

In [ ]:
# signal maps

so_types = ['type1', 'type2', 'type3']
so_mapnames = ['f090', 'f150']
so_map_dir = '/home/zatkins/scratch/projects/lat-iso/phase1/so_maps/sigurdkn/scantest/test4'

so_maps = [enmap.read_map(f'{so_map_dir}/{so_type}_all_3way_{s}_{so_mapname}_sky_map.fits') for so_type in so_types for so_mapname in so_mapnames for s in range(3)]
so_maps = enmap.samewcs(so_maps, so_maps[0])
so_maps = so_maps.reshape(len(so_types), len(so_mapnames), -1, *so_maps.shape[-3:])

In [ ]:
so_maps.shape, act_maps.shape

In [ ]:
plot_so_types = ['type1', 'type3']
plot_so_mapname = 'f090'

apod_deg = 1
box = [[-30, -12.5], [-20, -22.5]]

signal_so = {}
flat_signal_so = {}
for so_type in plot_so_types:

    t = so_types.index(so_type)
    f = so_mapnames.index(plot_so_mapname)

    signal_so[so_type], square_mask = get_2d_signal_ps(so_maps[t, f].squeeze(), box, mask, w_act, apod_deg=apod_deg)

    modlmap = square_mask.modlmap().astype(np.float32)
    bin_edges = np.arange(0, 20_000, 200)
    flat_signal_so[so_type] = get_flat_2d_ps(signal_so[so_type], modlmap, bin_edges)

signal_act, square_mask = get_2d_signal_ps(act_maps.squeeze(), box, mask, w_act, apod_deg=apod_deg)

modlmap = square_mask.modlmap().astype(np.float32)
bin_edges = np.arange(0, 20_000, 200)
flat_signal_act = get_flat_2d_ps(signal_act, modlmap, bin_edges)

In [ ]:
enplot.pshow(square_mask, ticks=1, colorbar=True, downgrade=2)

pol = 1

# center
for so_type in plot_so_types:
    print(so_type)
    enplot.pshow(utils.crop_center(enmap.fftshift(flat_signal_so[so_type][pol]), 30), ticks=[50, 90], colorbar=True, upgrade=(20, 20), min=-1, max=3)

# zoom out
downgrade = 4
for so_type in plot_so_types:
    print(so_type)
    enplot.pshow(utils.crop_center(enmap.fftshift(flat_signal_so[so_type][pol]), 600).downgrade(downgrade), ticks=[2000], colorbar=True, upgrade=(downgrade, downgrade), min=-1, max=3)

print('act pa5_f090')
enplot.pshow(utils.crop_center(enmap.fftshift(flat_signal_act[pol]), 30), ticks=[50, 90], colorbar=True, upgrade=(20, 20), min=-1, max=3)
enplot.pshow(utils.crop_center(enmap.fftshift(flat_signal_act[pol]), 600).downgrade(downgrade), ticks=[2000], colorbar=True, upgrade=(downgrade, downgrade), min=-1, max=3)